# Supervised learning: Spiking Neural Network

Welcome to the sixth laboratory class of MLHD course!

**Today you will see:**
- an implementation of a convolutional Spiking Neural Network to perform image classification with the MNIST dataset.

**After this assignment you will be able to:**
- build and train an SNN using snnTorch and PyTorch modules for a classification problem, using `nn.Module`.
- implement your own custom pipeline to train and evaluate the network.

**Note**: this notebook is adapted from [Tutorial 6 - Surrogate Gradient Descent in a Convolutional SNN](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_6.html).

**Reference paper**: *Jason K. Eshraghian, Max Ward, Emre Neftci, Xinxin Wang, Gregor Lenz, Girish Dwivedi, Mohammed Bennamoun, Doo Seok Jeong, and Wei D. Lu. "Training Spiking Neural Networks Using Lessons From Deep Learning". Proceedings of the IEEE, 111(9) September 2023* available [here](https://ieeexplore.ieee.org/abstract/document/10242251).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd '/content/drive/MyDrive/MLHD_labs/Lab_6'

In [ ]:
!pip install snntorch --quiet
!pip install torcheval --quiet

In [ ]:
import snntorch as snn
from snntorch import functional as SF
from snntorch import surrogate, utils, spikegen
from snntorch import spikeplot as splt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F
from torchsummary import summary
from torcheval.metrics.functional import binary_f1_score

import matplotlib.pyplot as plt
import numpy as np
import time
import tqdm
from matplotlib.animation import PillowWriter

from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_fscore_support, roc_curve, auc, accuracy_score, precision_recall_curve

torch.manual_seed(100)

## 1. Download the data and create Dataloaders

### 1.2 Download the MNIST dataset

In PyTorch, a *transform* defines a sequence of preprocessing steps that are automatically applied to each data sample when it is loaded from the dataset. This ensures that all images are processed in the same way before being passed to the model.

When loading the MNIST dataset from `torchvision`, we apply the following transformations:
- Resize each image to 28 x 28 pixels using `transforms.Resize((28, 28))`
- Convert to grayscale using `transforms.Grayscale()`
- Convert the image into a PyTorch tensor using `transforms.ToTensor()`
- Normalize the pixel values in [0,1] `using transforms.Normalize((0,), (1,))`

All these transformations are combined using `transforms.Compose()`, which takes a **list** of transformations and applies them sequentially, in the order they are listed.

In [ ]:
data_path ='/content/data/mnist'

In [ ]:
# Define a transform
# START CODE HERE (1 line)
transform = None
### END CODE HERE

# download the train and test set of MNSIST applying the transformations
mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)

In [ ]:
# plot some samples from the train set
fig, axes = plt.subplots(1, 4, figsize=(10, 3))

for i in range(4):
    image, label = mnist_train[i]
    image = image.squeeze()  
    
    axes[i].imshow(image, cmap='gray')
    axes[i].set_title(f"Label: {label}")
    axes[i].axis('off')

plt.show()

### 1.3 Create Dataloader

In PyTorch, a `DataLoader` is a utility that helps you efficiently load data in batches during training or evaluation. Instead of loading one sample at a time, the DataLoader groups data into mini-batches, can shuffle the data, and loads samples in parallel, which makes training faster and more organized.

To create a DataLoader, you first need a dataset, and then pass it to `torch.utils.data.DataLoader` (which is already imported as `DataLoader`), specifying also the `batch_size`, whether to `shuffle` the data (remember to shuffle just the train). Specify also `drop_last=True` for both dataloaders.

In [ ]:
batch_size = 128

# Create DataLoaders
# START CODE HERE (2 lines)
train_loader = None
test_loader = None
### END CODE HERE 

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using {device} device")

### 1.4 - Visualize encoded data

We now use *rate coding* to encode a batch of training data and visualize it. To do that, we use the function `spikegen.rate()`, which takes as input the data, and the parameter `num_steps`, which specifies the numer of timesteps. Set `num_steps=25`.

In [ ]:
# Iterate through minibatches
data, target = next(iter(train_loader))

# Spiking Data
# START CODE HERE (1 line)
spike_data = None
### END CODE HERE 
print(spike_data.size())

Expected output: `torch.Size([25, 128, 1, 28, 28])`

In [ ]:
# take on sample of the batch
spike_data_sample = spike_data[:, 0, 0]
print(spike_data_sample.size())

Run multiple time the following code to plot different timesteps of the same encoded image.

In [ ]:
timestep = rand_int = np.random.randint(0, 25) 
spike_img = spike_data_sample[timestep].cpu().numpy()  

plt.figure(figsize=(4, 4))
plt.imshow(spike_img, cmap='gray', interpolation='nearest')
plt.title(f"Spike Map - Timestep {timestep}")
plt.axis('off') 
plt.show()

We can also produce an animation of how the spikes evolve over time using `splt.animator`. The code below should create the animation and save it as a GIF.

In [ ]:
fig, ax = plt.subplots()
anim = splt.animator(spike_data_sample, fig, ax)
path = '/content/spike_encoding.gif'
anim.save(path, writer=PillowWriter(fps=20))

Don't worry if the plot does not display anything. The command `anim.save('spike_encoding.gif', writer=PillowWriter(fps=20))` saves the animation as a GIF, so that you can visualize it! (A copy of the GIF is also present in the drive folder)

**Note**: this was just to show you how you can use already implemented function from `snnTorch` to encode static data into spike trains. In the following, however, we will train our SNN classifier on static MNIST images, treating them as continuous input current.

## 2 - Model definition

The module `snn.Leaky` implements a first-order *leaky integrate-and-fire* neuron mode. It accepts the following parameters:

- `beta`: membrane potential decay.
- `threshold`: threshold for spike generation.
- `learn_beta`: option to enable learnable beta.
- `learn_threshold`: option to enable learnable threshold.
- `spike_grad`: surrogate gradient to use in the backward pass (see the [documentation](https://snntorch.readthedocs.io/en/latest/snntorch.surrogate.html)).
- `reset_mechanism`: defines the reset mechanism applied to the membrane potential each time a spike is generated.

We implementour SNN-based classifier using `nn.Module`.

The only parameter of the classifier is `timesteps` (number of timesteps of the SNN).

The classifier consists of:
- Layer 1:

    - Conv2d (use `nn.Conv2d`) with 1 input channel, 12 output channels, and kernel 5
    - Maxpool2d (use `nn.MaxPool2d`) with kernel 2
    - LIF layer (use `snn.Leaky`)

- Layer 2

    - Conv2d with 12 input channel, 32 output channels, and kernel 5
    - Maxpool2d with kernel 2
    - LIF layer 

- Layer output

    - flatten layer (use `nn.Flatten()`)
    - Fully-connected layer (use `nn.Linear`) with $32\times4\times4$ input features and 10 output features
    - LIF layer 

**Note**: For each LIF layer, we just specify `beta = 0.5` and `spike_grad = surrogate.fast_sigmoid(slope=25)`. Fill free to change the values of both parameters and specify also the threshold! For more info about LIF layer, see the [documentation](https://snntorch.readthedocs.io/en/latest/snn.neurons_leaky.html).

During the *forward pass* of the SNN:

- First, initialize the membrane potentials for each LIF layer. This sets the neurons’ initial states at the beginning of the sequence. Use `name_layer.init_leaky()` (example: `mem1 = self.lif1.init_leaky()`)
- Then, for each timestep, run the network to simulate its temporal dynamics. At each layer, the current (which comes from a linear or convolutional layer followed by maxpool) is fed into the LIF neurons.
- The LIF layer uses this current along with its previous membrane potential to produce spikes and update its membrane potential.
- For the first layer, the input current is the raw data. For all subsequent layers, the input comes from the spikes generated by the previous layer.

This process allows the network to propagate information over time while updating neuron states at each timestep.

The dynamics of the first layer are already implemented and can be used as a template for the two subsequent layers:

```python
cur1 = self.pool1(self.conv1(x))
spk1, mem1 = self.lif1(cur1, mem1)
```

In [ ]:
class SNN(nn.Module):
    def __init__(self, timesteps):
        super().__init__()

        self.timesteps = timesteps

        # first layer
        # START CODE HERE ### (3 lines)
        self.conv1 = None
        self.pool1 = None
        self.lif1 = None
        ### END CODE HERE ###

        # second layer
        # START CODE HERE ### (3 lines)
        self.conv2 = None
        self.pool2 = None
        self.lif2 = None
        ### END CODE HERE ###

        # ouput layer
        # START CODE HERE ### (3 lines)
        self.flatten = None
        self.fc1 = None
        self.lif3 = None
        ### END CODE HERE ###

    def forward(self, x):

        # Initialize membrane potentials of LIF neurons
        # START CODE HERE ### (3 lines)
        mem1 = None
        mem2 = None
        mem3 = None
        ### END CODE HERE ###

        mem_rec = []
        spk_rec = []

        for step in range(self.timesteps):
            
            # firt layer
            cur1 = self.pool1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)

            # second layer
            # START CODE HERE ### (2 lines)
            cur2 = None
            spk2, mem2 = None
            ### END CODE HERE ###

            # output layer
            # START CODE HERE ### (2 lines)
            cur3 = None
            spk_out, mem_out = None
            ### END CODE HERE ###

            # keep track of output spikes and membrane potential
            spk_rec.append(spk_out)
            mem_rec.append(mem_out)

        return torch.stack(spk_rec, dim=0), torch.stack(mem_rec, dim=0)

**Important note**: in **classification tasks**, the spikes from the last layer are typically accumulated over time into a list (`spk_rec.append(spk_out)`), and then converted to a tensor of shape $(T, B, C)$, where $T$ is the number of timesteps, $B$ is the batch size, and $C$ is the number of classes, with `torch.stack(spk_rec, dim=0)`.

Next, create an instance of the `SNN` model, called `classifier`, specifying `timesteps=25`, and move the classifier to the device you are using, with `classifier = classifier.to(device)`.

In [ ]:
# create an instance of the SNN model 
# START CODE HERE (1 line)
classifier = None
### END CODE HERE 

# move the classifier to the specific device
# START CODE HERE (1 line)
classifier = None
### END CODE HERE 

# print a summary of the model (note: summary requires the input size and the batch size)
summary(classifier, input_size = (1, 28, 28), batch_size = batch_size)

 ## 3 - Model training and testing
 ### 3.1 - Define the training and evaluating function

In PyTorch, training a neural network is explicit and manual. Unlike TensorFlow/Keras, there is no `model.fit()` function that automatically handles the training loop. Instead, we write our own training code, which gives us more flexibility and control over what happens at each step.

Inside the function `train_fn`:

1. Set the model in training mode before starting the training loop using `model.train()`
2. Compute a forward pass, passing the input data through the model to get predictions. Remember that the SNN classifier returns both the spikes and the membrane potential (hint: use  `spk_out, mem_out = model(X.float())`)
3. Reset all gradients to zero. PyTorch accumulates gradients by default, so if we don’t zero them, the gradients from previous batches would incorrectly affect the current batch. This is done by calling `optimizer.zero_grad()`.
4. Compute the loss between the model's output spikes and the target *y*
5. Perform a backward pass using `loss.backward()`
6. Update the model’s weights using the optimizer. Call `optimizer.step()` applies the gradients computed during the backward pass and performs a step of gradient descent
7. When evaluating the model on a test set, remember to set `model.eval()`

During the backward pass, PyTorch uses `torch.autograd`, an automatic differentiation engine that computes gradients for all learnable parameters.

In [ ]:
def train_fn(model, train_loader, test_loader, accuracy, loss_fn, optimizer, 
             epochs, test_every=50, path=None, verbose=True):

    N_train = len(train_loader)
    # create lists to store loss and accuracy across epochs
    train_loss_list, test_acc_list = [], []

    # set the model in training phase
    # START CODE HERE ### (1 line)
    None
    ### END CODE HERE ###

    counter = 0

    for epoch in range(epochs):
        start_time = time.time()

        train_loss = 0.0

        for (X, y) in tqdm.tqdm(iter(train_loader)):
            # move tensors to device
            X, y = X.to(device), y.to(device)

            # forward pass
            # START CODE HERE ### (1 line)
            None
            ### END CODE HERE ###

            # zero the gradients
            # START CODE HERE ### (1 line)
            None
            ### END CODE HERE ###

            # compute the loss 
            # START CODE HERE ### (1 line)
            loss = None
            ### END CODE HERE ###

            # backward pass
            # START CODE HERE ### (1 line)
            None
            ### END CODE HERE ###

            # update model parameters through the optimizer
            # START CODE HERE ### (1 line)            
            None
            ### END CODE HERE ###

            # accumulate loss
            train_loss += loss.item()
            
            # Test set evaluation every 'test_every' iterations
            if counter % test_every == 0:
                
                with torch.no_grad():
                    # set the model in evaluation mode
                    # START CODE HERE ### (1 line)            
                    None
                    ### END CODE HERE ###

                    # compute accuracy on the test set
                    total_acc = 0.0

                    for X_test, y_test in test_loader:
                        # move tensors to device
                        X_test, y_test = X_test.to(device), y_test.to(device)

                        spk_out_test, _ = model(X_test.float())
                        total_acc += accuracy(spk_out_test, y_test).item()

                    avg_test_acc = total_acc / len(test_loader)
                    test_acc_list.append(avg_test_acc)

                    if verbose:
                        print(f"Iteration {counter}, Test Acc: {avg_test_acc * 100:.2f}%")
                    
                    # switch back to training mode
                    model.train() 

            counter += 1

        train_loss_list.append(train_loss / N_train)

        torch.cuda.empty_cache()
        end_time = time.time()

        if verbose:
            print(f"Epoch {epoch+1}/{epochs} - {int(end_time-start_time)}s - "
                  f"loss: {round(train_loss_list[-1], 4)}")
        
    if path:
        torch.save(model.state_dict(), path)

    return train_loss_list, test_acc_list

### 3.2 - Train the network

In this part you will specify

- the optimizer: in this lab, we use `torch.optim.Adam(classifier.parameters(), lr = 1e-2, betas=(0.9, 0.999))`)
- the loss function: in this lab, we use `SF.ce_rate_loss()`. This applies the cross entropy loss to the output *spike count* in order train a *rate-coded* network.
[See the documentation for further information and exmaples.](https://snntorch.readthedocs.io/en/latest/snntorch.functional.html#snntorch.functional.ce_rate_loss)
- the accuracy metric:  in this lab, we use `SF.acc.accuracy_rate`. This function works similarly, in that the predicted output spikes and actual targets are supplied as arguments. `accuracy_rate` assumes a **rate code** is used to interpret the output by checking if the index of the neuron with the **highest spike count** matches the target index. Note that using this function is the same as doing
```python
# Extract spike output shape
T, B, C = spk_out.shape
# Sum spikes over time
spike_sum = spk_out.sum(dim=0)  # shape becomes (B, C)
# Get predicted class (neuron with highest spike count)
pred_class = spike_sum.argmax(dim=1)  # shape (B,)
# Compare with target and compute number of correct predictions
correct = (pred_class == target).sum().item()
# Compute average accuracy of batch
accuracy = correct / B
```

**Note**: a variety of loss functions are included in the `snn.functional` module, which is analogous to `torch.nn.functional` in PyTorch.
These implement a mix of cross entropy and mean square error losses, are applied to spikes and/or membrane potential, to train a rate or latency-coded network.

In [ ]:
# the network is slow, run just for one epoch
num_epochs = 1

# specify the optimizer
# START CODE HERE (1 line)
optimizer = None
### END CODE HERE ###

# specify the correct loss function and accuracy metric to be used
# START CODE HERE (2 lines)
loss =  None
acc = None
### END CODE HERE ###

# train the network (takes ~10 min)
train_loss, test_acc = train_fn(model = classifier,
                                train_loader = train_loader,
                                test_loader= test_loader,
                                accuracy = acc,
                                loss_fn = loss,
                                optimizer = optimizer,
                                epochs = num_epochs,
                                path = 'snn_model.pt'
                                )


In [ ]:
# Plot Loss
fig = plt.figure()
plt.plot(test_acc)
plt.title("Test Set Accuracy")
plt.grid(alpha=.5)
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.show()

## 4 - Compute some validation metrics

If you do not want to wait for the training to finish, you can load some pretrained models.

Just uncomment the lines of code below.

In [ ]:
#classifier = SNN(timesteps=25).to(device)
#classifier.load_state_dict(torch.load("snn_model.pt", map_location=torch.device(device)))
#summary(classifier, input_size = (1, 28, 28), batch_size = batch_size)

### 4.1 - Evaluate the classifier on the test set (compute accuracy, precision, recall, and F1 score)

In [ ]:
def evaluate(model, data_loader):
    N_test = len(data_loader)
    clss_list, y_list = [], []
    
    with torch.no_grad():
      model.eval()
      acc = 0.0

      for X, y in tqdm.tqdm(iter(data_loader)):
          
          X = X.to(device)

          spk_out, _ = model(X.float())

          clss = torch.argmax(torch.sum(spk_out, 0), dim=1)
          acc += SF.acc.accuracy_rate(spk_out, y).item()

          clss_list.extend(clss.cpu())
          y_list.extend(y.cpu())

      tot_acc = acc / N_test

      return tot_acc, clss_list, y_list

In [ ]:
# evaluate the model
test_acc, test_preds, test_y = evaluate(classifier, test_loader)

Uses sklearn’s `precision_recall_fscore_support` function (already imported) between `test_y` and `test_preds`, specifying `average='macro'` (meaning that each metric is computed independently for each class and then averaged).

In [ ]:
# Evaluate precision, recall and fscore
### START CODE HERE ### (1 line)
precision, recall, fscore, _= None
### END CODE HERE ###

print('#### TEST PERFORMANCE\n')
print('Accuracy: {:.2f}%'.format(test_acc*100))
print('Precision: {:.2f}%'.format(precision*100))
print('Recall: {:.2f}%'.format(recall*100))
print('Fscore: {:.2f}%'.format(fscore*100))

### 4.2 - Plot Receiver operating characteristic (ROC) curve for each class

In [ ]:
# Convert labels to one-hot encoding
y_onehot = label_binarize(test_y, classes=range(10))
preds = label_binarize(test_preds,  classes=range(10)) 

plt.figure(figsize=(8, 6))

for i in range(10):
    fpr, tpr, _ = roc_curve(y_onehot[:, i], preds[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {i} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for MNIST Classes')
plt.legend(loc='lower right')
plt.grid(alpha=.5)
plt.show()

## 5 - Visualize spike counter of the output layer

Run a forward pass on a batch of data to obtain spike and membrane potentials of the last layer.

In [ ]:
# Get a single batch from the test loader
X_test, y_test = next(iter(test_loader))

# Move to device (CPU or GPU)
X_test = X_test.to(device)
y_test = y_test.to(device)

# Forward pass through the classifier
spk_rec, mem_rec = classifier(X_test.float())

Changing `idx` allows you to index into various samples from the simulated minibatch. Use `splt.spike_count` to explore the spiking behaviour of a few different samples!

The line `anim.save('animation.gif', writer=PillowWriter(fps=20))` will save the animation as a GIF, to visualize it.



In [ ]:
idx = 18

fig, ax = plt.subplots(facecolor='w', figsize=(12, 7))
labels=['0', '1', '2', '3', '4', '5', '6', '7', '8','9']
print(f"The target label is: {y_test[idx]}")

#  Plot spike count histogram
anim = splt.spike_count(spk_rec[:, idx].detach().cpu(), fig, ax, labels=labels,
                        animate=True, interpolate=4)

path = '/content/spike_count.gif'
anim.save(path, writer=PillowWriter(fps=20))

Congratulations! Lab 6 successfully completed :)